**Question:** Which point-in-time features can be built without leakage, and how are they tagged as ELO / ELO-substitute / neutral?

**Assumptions:** Every feature uses only information strictly before the match date (documented roster/signup exemption).

**Method:** Derive features, audit leakage, expand each match into two perspective rows, and mark an 80/20 chronological split.

**Result:** A leakage-audited, tagged, symmetric frame (13,702 train / 3,426 test matches) saved for all later notebooks.

# 02 — Feature Engineering, Leakage Audit, Symmetry

Builds on the EDA and data-quality decisions from `01_data_exploration.ipynb` (Godiste bounds
`[1930, 2015]`, clean `Lokacija`, ~70% H2H sparsity).

**Pipeline in this notebook:**
1. Derive point-in-time, notebook-only features (season match count, home-court share, age,
   STB/Decider clutch win rates, relative pyramid position).
2. Leakage audit (every feature checked against the strict `< Datum` rule or the documented
   roster/signup exemption) + ELO / ELO-substitute / neutral tagging.
3. Duplicate-and-flip symmetric row expansion.
4. Time-based train/test split marker.
5. Save the final frame to `../data/features.parquet` (gitignored — contains real `IgracId`s)
   plus the tagging dict to `../data/feature_tags.json`.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'matches_export.csv', parse_dates=['Datum'])
df['Datum'] = pd.to_datetime(df['Datum']).dt.tz_localize(None)
# Global chronological order, SusretId as tiebreak -- same convention as the SQL export's
# ORDER BY Datum ASC, SusretId ASC (Part 1.5).
df = df.sort_values(['Datum', 'SusretId']).reset_index(drop=True)
print(df.shape)
df.head(2)


## Step 1 — Long-format player-match log

Needed for point-in-time PER-PLAYER window features (season match count, home-court share, clutch win rates). Every match contributes two rows: one from the winner's perspective, one from the loser's — this mirrors the SQL export's `PlayerMatchLog` CTE, just done in pandas -- duplicate-and-flip-style expansion is a notebook step, not part of the SQL export.

In [ ]:
common_cols = ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'LigaTip', 'LigaKoeficjent', 'Lokacija']

winner_rows = df[common_cols + ['WinnerId', 'LoserId', 'W_Set1', 'W_Set2', 'W_Set3', 'L_Set1', 'L_Set2', 'L_Set3']].rename(
    columns={'WinnerId': 'IgracId', 'LoserId': 'OpponentId',
             'W_Set1': 'MySet1', 'W_Set2': 'MySet2', 'W_Set3': 'MySet3',
             'L_Set1': 'OppSet1', 'L_Set2': 'OppSet2', 'L_Set3': 'OppSet3'}
).assign(Won=1)

loser_rows = df[common_cols + ['LoserId', 'WinnerId', 'L_Set1', 'L_Set2', 'L_Set3', 'W_Set1', 'W_Set2', 'W_Set3']].rename(
    columns={'LoserId': 'IgracId', 'WinnerId': 'OpponentId',
             'L_Set1': 'MySet1', 'L_Set2': 'MySet2', 'L_Set3': 'MySet3',
             'W_Set1': 'OppSet1', 'W_Set2': 'OppSet2', 'W_Set3': 'OppSet3'}
).assign(Won=0)

player_log = pd.concat([winner_rows, loser_rows], ignore_index=True)
player_log['LokacijaNorm'] = player_log['Lokacija'].str.strip().str.lower()
# CRITICAL: sort strictly ascending by (IgracId, Datum, SusretId) -- every expanding/cumcount
# feature below only ever looks at rows ABOVE the current one in this per-player ordering, i.e.
# strictly BEFORE this match. This is the same discipline as the SQL ROW_NUMBER/LAG pattern.
player_log = player_log.sort_values(['IgracId', 'Datum', 'SusretId']).reset_index(drop=True)
print(player_log.shape)
player_log.head(4)


## Step 2 — Point-in-time per-player features (strictly BEFORE this row)

In [ ]:
g_player = player_log.groupby('IgracId', sort=False)

# All-time match count before this match (cross-checked against the SQL export below)
player_log['MatchCountBefore_check'] = g_player.cumcount()
# Matches played THIS season before this match (WinnerSeasonMatchCount/LoserSeasonMatchCount --
# 'deliberately deferred to pandas' per Part 1.5, now trivial via the exported SezonaId)
player_log['SeasonMatchCountBefore'] = player_log.groupby(['IgracId', 'SezonaId'], sort=False).cumcount()
# Matches played at THIS location before this match (raw ingredient for home-court share)
player_log['HomeCountBefore'] = player_log.groupby(['IgracId', 'LokacijaNorm'], sort=False).cumcount()

# Cross-check: does our pandas point-in-time count match the SQL-exported
# WinnerMatchCountBefore/LoserMatchCountBefore exactly? If not, our ordering assumption is wrong.
check_w = df[['SusretId', 'WinnerId', 'WinnerMatchCountBefore']].rename(
    columns={'WinnerId': 'IgracId', 'WinnerMatchCountBefore': 'ExportedCount'})
check_l = df[['SusretId', 'LoserId', 'LoserMatchCountBefore']].rename(
    columns={'LoserId': 'IgracId', 'LoserMatchCountBefore': 'ExportedCount'})
check_all = pd.concat([check_w, check_l])
merged_check = player_log.merge(check_all, on=['SusretId', 'IgracId'], how='left')
mismatches = (merged_check['MatchCountBefore_check'] != merged_check['ExportedCount']).sum()
print(f"Cross-check MatchCountBefore (pandas) vs SQL-exported count: {mismatches} mismatches "
      f"out of {len(merged_check)} rows")
if mismatches:
    print(merged_check.loc[merged_check['MatchCountBefore_check'] != merged_check['ExportedCount']].head(10))
assert mismatches == 0, "pandas point-in-time ordering does not match the SQL export's ROW_NUMBER ordering!"
print("OK -- pandas (IgracId, Datum, SusretId) ordering exactly reproduces the SQL ROW_NUMBER window.")


### Home-court share

In [ ]:
player_log['HomeShareBefore'] = np.where(
    player_log['MatchCountBefore_check'] > 0,
    player_log['HomeCountBefore'] / player_log['MatchCountBefore_check'],
    np.nan,
)
print(player_log['HomeShareBefore'].describe())
print(f"NaN (player's debut match -- zero match history, no home-court signal yet): "
      f"{player_log['HomeShareBefore'].isna().sum()} ({player_log['HomeShareBefore'].isna().mean():.2%})")


### Clutch: STB / Decider win rates

`STBWinRateBefore`/`STBCountBefore` (super tie-break: 3rd set present AND either side's `Set3 >= 10`)
and `DeciderWinRateBefore`/`DeciderCountBefore` (any 3rd set). Both computed over the player's
PRIOR matches ONLY -- whether THIS match goes to a decider/STB is an outcome of this match and is
never used as an input feature for predicting this match (this is exactly the leakage trap this
project's point-in-time discipline guards against). Sparsity handled via shrinkage toward 0.5 with a small prior weight,
raw counts kept as separate reliability features.


In [ ]:
player_log['Is3rdSet'] = player_log['MySet3'].notna()
player_log['IsSTB'] = player_log['Is3rdSet'] & ((player_log['MySet3'] >= 10) | (player_log['OppSet3'] >= 10))
player_log['STBWin'] = (player_log['IsSTB'] & (player_log['Won'] == 1)).astype(int)
player_log['DeciderWin'] = (player_log['Is3rdSet'] & (player_log['Won'] == 1)).astype(int)
player_log['IsSTB_int'] = player_log['IsSTB'].astype(int)
player_log['Is3rdSet_int'] = player_log['Is3rdSet'].astype(int)

g_player = player_log.groupby('IgracId', sort=False)
# cumsum() INCLUDES the current row -- subtract the current row's own value to get
# "sum of all STRICTLY PRIOR rows", the same trick as the SQL ROWS BETWEEN ... 1 PRECEDING window.
player_log['STBCountBefore'] = g_player['IsSTB_int'].cumsum() - player_log['IsSTB_int']
player_log['STBWinsBefore'] = g_player['STBWin'].cumsum() - player_log['STBWin']
player_log['DeciderCountBefore'] = g_player['Is3rdSet_int'].cumsum() - player_log['Is3rdSet_int']
player_log['DeciderWinsBefore'] = g_player['DeciderWin'].cumsum() - player_log['DeciderWin']

CLUTCH_PRIOR = 3  # small shrinkage weight toward 0.5 -- same idea as a Bayesian prior
player_log['STBWinRateBefore'] = (player_log['STBWinsBefore'] + CLUTCH_PRIOR * 0.5) / (player_log['STBCountBefore'] + CLUTCH_PRIOR)
player_log['DeciderWinRateBefore'] = (player_log['DeciderWinsBefore'] + CLUTCH_PRIOR * 0.5) / (player_log['DeciderCountBefore'] + CLUTCH_PRIOR)

print(f"Matches with a 3rd set at all: {player_log['Is3rdSet'].mean():.1%}")
print(f"Matches that were STBs: {player_log['IsSTB'].mean():.1%}")
print()
print(player_log[['STBCountBefore', 'STBWinRateBefore', 'DeciderCountBefore', 'DeciderWinRateBefore']].describe())


## Step 3 — Merge per-player point-in-time features back onto the match-level frame

In [ ]:
feature_cols = [
    'SusretId', 'IgracId', 'SeasonMatchCountBefore', 'HomeShareBefore',
    'STBCountBefore', 'STBWinRateBefore', 'DeciderCountBefore', 'DeciderWinRateBefore',
]

winner_feats = player_log.loc[player_log['Won'] == 1, feature_cols].rename(
    columns={c: f'Winner{c}' for c in feature_cols if c not in ('SusretId', 'IgracId')}
).rename(columns={'IgracId': 'WinnerId'})

loser_feats = player_log.loc[player_log['Won'] == 0, feature_cols].rename(
    columns={c: f'Loser{c}' for c in feature_cols if c not in ('SusretId', 'IgracId')}
).rename(columns={'IgracId': 'LoserId'})

df = df.merge(winner_feats, on=['SusretId', 'WinnerId'], how='left', validate='one_to_one')
df = df.merge(loser_feats, on=['SusretId', 'LoserId'], how='left', validate='one_to_one')
print(df.shape)
new_cols = [c for c in df.columns if any(k in c for k in
            ('SeasonMatchCountBefore', 'HomeShareBefore', 'STBWinRateBefore', 'DeciderWinRateBefore'))]
df[new_cols].describe()


## Step 4 — Age from `Godiste` (QC bounds decided in `01_data_exploration.ipynb`)

`Igrac.DatumRodjenja` is 100% NULL in production (see `../data/README.md`) -- `Godiste` (birth
year int) is the real, usable field. Bounds `[1930, 2015]` null out the 5 known placeholder rows
(`0`, `42`, `81`, `7101976`, `21021985`) while keeping every real value (min 1947, max 2007).


In [ ]:
GODISTE_LOWER, GODISTE_UPPER = 1930, 2015

def clean_godiste(s):
    return s.where((s >= GODISTE_LOWER) & (s <= GODISTE_UPPER))

if 'WinnerGodiste' in df.columns:
    df['WinnerGodisteClean'] = clean_godiste(df['WinnerGodiste'])
    df['LoserGodisteClean'] = clean_godiste(df['LoserGodiste'])

    df['WinnerAge'] = df['Datum'].dt.year - df['WinnerGodisteClean']
    df['LoserAge'] = df['Datum'].dt.year - df['LoserGodisteClean']

    n_dropped_w = (df['WinnerGodiste'].notna() & df['WinnerGodisteClean'].isna()).sum()
    n_dropped_l = (df['LoserGodiste'].notna() & df['LoserGodisteClean'].isna()).sum()
    print(f"Nulled out as implausible: Winner {n_dropped_w}, Loser {n_dropped_l}")
else:
    # Pseudonymised export (scripts/anonimiziraj.py): birth year was already converted to
    # age-at-match (same [1930, 2015] QC applied there) and then dropped -- nothing to compute.
    assert {'WinnerAge', 'LoserAge'} <= set(df.columns)
print(df[['WinnerAge', 'LoserAge']].describe())


## Step 5 — Home-share diff + Elo-confidence meta-feature (Part 2.2 table)

In [ ]:
df['WinnerEloConfidence'] = df['WinnerMatchCountBefore'].clip(upper=30) / 30
df['LoserEloConfidence'] = df['LoserMatchCountBefore'].clip(upper=30) / 30
df[['WinnerEloConfidence', 'LoserEloConfidence']].describe()


## Step 6 — H2H win rate (shrunk) from the exported raw H2H counts

`H2HWinsWinnerBefore` + `H2HWinsLoserBefore` == `H2HTotalBefore` by construction (every prior
meeting between this exact pair has exactly one winner, no draws in tennis) -- the symmetric
shrinkage below is verified to sum to exactly 1.0 for both players as a sanity check.


In [ ]:
consistency = (df['H2HWinsWinnerBefore'].fillna(0) + df['H2HWinsLoserBefore'].fillna(0) - df['H2HTotalBefore']).abs()
assert (consistency < 1e-9).all(), "H2H win counts do not sum to H2HTotalBefore!"

H2H_PRIOR = 2  # small shrinkage weight toward 0.5
df['WinnerH2HWinRateBefore'] = (df['H2HWinsWinnerBefore'].fillna(0) + H2H_PRIOR * 0.5) / (df['H2HTotalBefore'] + H2H_PRIOR)
df['LoserH2HWinRateBefore'] = (df['H2HWinsLoserBefore'].fillna(0) + H2H_PRIOR * 0.5) / (df['H2HTotalBefore'] + H2H_PRIOR)

sum_check = (df['WinnerH2HWinRateBefore'] + df['LoserH2HWinRateBefore'])
print(sum_check.describe())
assert np.allclose(sum_check, 1.0), "Shrunk H2H win rates should sum to exactly 1.0 by construction!"
print("OK -- WinnerH2HWinRateBefore + LoserH2HWinRateBefore == 1.0 for every row.")


## Step 7 — Relative pyramid position (razina normalized)

**Caveat (see notebook 01):** `BrojLigaUSezoni` counts distinct `LigaId` *groups* that season, not
skill *levels* (`LigaKoeficjent` only spans 1-8). So `LigaKoeficjent / BrojLigaUSezoni` (the
originally planned literal ratio) is a rough depth-normalized proxy, not a clean percentile. We
compute that literal ratio AND a cleaner empirical percentile (`RelativeRazinaPct`, using the
actual min/max `LigaKoeficjent` observed that season+gender), and keep both, clearly labeled, so
the modeling notebooks can compare which is more predictive.

In [ ]:
df['RelativeRazina_ByLigaCount'] = df['LigaKoeficjent'] / df['BrojLigaUSezoni']
df['RelativeRazina_ByPlayerCount'] = df['LigaKoeficjent'] / df['BrojIgracaUkupno']

season_gender_koef = df.groupby(['SezonaId', 'LigaTip'])['LigaKoeficjent'].agg(['min', 'max']).rename(
    columns={'min': 'KoefMin', 'max': 'KoefMax'})
df = df.merge(season_gender_koef, on=['SezonaId', 'LigaTip'], how='left')
span = (df['KoefMax'] - df['KoefMin']).replace(0, np.nan)
df['RelativeRazinaPct'] = 1 - (df['LigaKoeficjent'] - df['KoefMin']) / span
df['RelativeRazinaPct'] = df['RelativeRazinaPct'].fillna(1.0)  # only one level that season -> top=1.0
df = df.drop(columns=['KoefMin', 'KoefMax'])

# Debut-razina analog (asymmetric -- per player's own debut season/gender context)
df['WinnerDebutRelativeRazina'] = df['WinnerDebutRazina'] / df['WinnerDebutBrojLiga']
df['LoserDebutRelativeRazina'] = df['LoserDebutRazina'] / df['LoserDebutBrojLiga']

df[['RelativeRazina_ByLigaCount', 'RelativeRazina_ByPlayerCount', 'RelativeRazinaPct',
    'WinnerDebutRelativeRazina', 'LoserDebutRelativeRazina']].describe()


## Step 8 — Leakage-audit checklist (mandatory, Part 2.2/2.2.1)

One row per **conceptual** feature (the base name that will drive the `_diff` / symmetric column
in the duplicate-and-flip step below). For every feature: (1) point-in-time rule or the documented
roster/signup exemption, (2) does it read any column from the row being predicted (must be False
for all), (3) ELO / ELO-substitute / neutral tag.


In [ ]:
FEATURE_AUDIT = [
    dict(feature='elo', pit='SusretIgracElo.EloBefore, point-in-time by construction (EloRegenerate walks matches chronologically)',
         reads_outcome=False, tag='ELO', note='Exp B only; excluded entirely from Exp A'),
    dict(feature='match_count_before', pit="ROW_NUMBER() OVER (PARTITION BY IgracId ORDER BY Datum,SusretId) - 1",
         reads_outcome=False, tag='neutral', note='experience, not opponent quality'),
    dict(feature='elo_confidence', pit='fn(match_count_before) = min(count,30)/30',
         reads_outcome=False, tag='neutral', note='Elo-reliability meta-signal'),
    dict(feature='razina_match_count_before', pit='ROW_NUMBER PARTITION BY IgracId, LigaKoeficjent',
         reads_outcome=False, tag='neutral', note='experience at this specific razina'),
    dict(feature='days_since_last_match', pit='LAG(Datum) OVER (PARTITION BY IgracId ORDER BY Datum,SusretId), strictly prior',
         reads_outcome=False, tag='neutral', note='rust/fitness proxy; NaN on debut match'),
    dict(feature='forma_last10', pit='AVG(Won) OVER ROWS BETWEEN 10 PRECEDING AND 1 PRECEDING',
         reads_outcome=False, tag='ELO-substitute', note='rolling win-rate form'),
    dict(feature='career_winrate_before', pit='AVG(Won) OVER ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING',
         reads_outcome=False, tag='ELO-substitute', note='built ONLY from Won, never from EloBefore/EloAfter'),
    dict(feature='h2h_win_rate_before', pit='H2HWinsBefore/H2HTotalBefore, COUNT window over strictly-prior meetings',
         reads_outcome=False, tag='ELO-substitute', note='shrunk toward 0.5 (prior weight=2)'),
    dict(feature='h2h_total_before', pit='COUNT window over strictly-prior meetings',
         reads_outcome=False, tag='neutral', note='symmetric; reliability count, not an outcome signal'),
    dict(feature='liga_koeficjent', pit="this match's own LigaKoeficjent -- administratively assigned before the match is played",
         reads_outcome=False, tag='ELO-substitute', note='symmetric; skill-tier proxy independent of the numeric Elo rating'),
    dict(feature='liga_tip', pit='administrative category (== gender in this singles-only scope)',
         reads_outcome=False, tag='neutral', note='symmetric; low-signal category'),
    dict(feature='broj_liga_u_sezoni', pit='SezonaLigaIgrac roster, FIXED at season start -- roster/signup exemption (Part 2.2.1 addendum), not derived from match outcomes',
         reads_outcome=False, tag='neutral', note='symmetric'),
    dict(feature='broj_igraca_ukupno', pit='same roster exemption as broj_liga_u_sezoni',
         reads_outcome=False, tag='neutral', note='symmetric'),
    dict(feature='broj_igraca_uligi', pit='same roster exemption, scoped to this specific LigaId',
         reads_outcome=False, tag='neutral', note='symmetric'),
    dict(feature='relative_razina_by_liga_count', pit='fn(liga_koeficjent, broj_liga_u_sezoni), both point-in-time safe',
         reads_outcome=False, tag='ELO-substitute', note='symmetric; inherits tag from liga_koeficjent; see BrojLigaUSezoni=groups-not-levels caveat'),
    dict(feature='relative_razina_by_player_count', pit='fn(liga_koeficjent, broj_igraca_ukupno)',
         reads_outcome=False, tag='ELO-substitute', note='symmetric; same caveat'),
    dict(feature='relative_razina_pct', pit='fn(liga_koeficjent, season+gender min/max koef -- both roster/admin, not outcome-derived)',
         reads_outcome=False, tag='ELO-substitute', note='symmetric; cleaner 0-1 percentile alternative'),
    dict(feature='debut_razina', pit='FIRST_VALUE OVER (PARTITION BY IgracId ORDER BY Datum,SusretId ROWS UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) -- always equals the debut match\'s own PRE-ASSIGNED razina, never an outcome (Part 1.5/2.2 addendum)',
         reads_outcome=False, tag='ELO-substitute', note='admin/wild-card skill judgment made before any Elo exists for that player'),
    dict(feature='debut_broj_liga', pit='roster exemption, joined on the player\'s debut season/gender',
         reads_outcome=False, tag='neutral', note='normalizes debut razina across eras'),
    dict(feature='debut_broj_igraca', pit='roster exemption, joined on the player\'s debut season/gender',
         reads_outcome=False, tag='neutral', note='normalizes debut razina across eras'),
    dict(feature='debut_relative_razina', pit='fn(debut_razina, debut_broj_liga)',
         reads_outcome=False, tag='ELO-substitute', note='inherits tag from debut_razina'),
    dict(feature='season_match_count_before', pit='pandas cumcount PARTITION BY IgracId,SezonaId ORDER BY Datum,SusretId (strict; cross-checked vs SQL all-time count in Step 2)',
         reads_outcome=False, tag='neutral', note='season-specific fitness/activity'),
    dict(feature='home_share_before', pit='pandas cumcount PARTITION BY IgracId,Lokacija / cumcount PARTITION BY IgracId, both strictly prior',
         reads_outcome=False, tag='neutral', note='NaN on player\'s debut match'),
    dict(feature='age', pit="YEAR(Datum) - Godiste; Godiste fixed at signup, predates every match by construction",
         reads_outcome=False, tag='neutral', note='QC-bounded [1930,2015], see notebook 01'),
    dict(feature='stb_win_rate_before', pit='cumsum-minus-self PARTITION BY IgracId, strictly prior rows only',
         reads_outcome=False, tag='ELO-substitute', note='whether THIS match is an STB is an outcome and is NEVER used'),
    dict(feature='stb_count_before', pit='same as stb_win_rate_before',
         reads_outcome=False, tag='neutral', note='reliability count'),
    dict(feature='decider_win_rate_before', pit='cumsum-minus-self PARTITION BY IgracId, strictly prior rows only',
         reads_outcome=False, tag='ELO-substitute', note='whether THIS match is a decider is an outcome and is NEVER used'),
    dict(feature='decider_count_before', pit='same as decider_win_rate_before',
         reads_outcome=False, tag='neutral', note='reliability count'),
]

audit_df = pd.DataFrame(FEATURE_AUDIT)
assert audit_df['reads_outcome'].sum() == 0, "A feature reads the outcome of the match being predicted!"
assert audit_df['feature'].is_unique, "Duplicate feature name in the audit table!"
print(f"{len(audit_df)} features audited, 0 read the match outcome, all have a documented point-in-time rule.")
audit_df


In [ ]:
FEATURE_TAGS = dict(zip(audit_df['feature'], audit_df['tag']))
tag_counts = audit_df['tag'].value_counts()
print(tag_counts)
FEATURE_TAGS


## Step 9 — Duplicate-and-flip symmetric expansion (Part 2.3)

For every match, create TWO rows: one from the perspective of the actual winner
(`Label=1`), one from the perspective of the actual loser (`Label=0`). Asymmetric features are
expressed as `(this_player - opponent)` diffs, which negate exactly when the perspective flips --
this is what guarantees `P(A beats B) + P(B beats A) = 1` by construction. Symmetric
match-level context (`h2h_total_before`, `liga_koeficjent`, the population/pyramid columns) does
not flip -- identical on both rows.

Raw per-perspective values (`player_<x>`/`opponent_<x>`) are also kept, but only for audit/
debugging -- they are NOT part of the intended P3 model-input set (`MODEL_INPUT_COLS` below),
since the diff already carries the signal and duplicating it as two raw values would just
reintroduce absolute-scale noise the diff was designed to remove.


In [ ]:
ASYMMETRIC_PAIRS = [
    ('elo', 'WinnerEloBefore', 'LoserEloBefore'),
    ('match_count_before', 'WinnerMatchCountBefore', 'LoserMatchCountBefore'),
    ('elo_confidence', 'WinnerEloConfidence', 'LoserEloConfidence'),
    ('razina_match_count_before', 'WinnerRazinaMatchCountBefore', 'LoserRazinaMatchCountBefore'),
    ('days_since_last_match', 'WinnerDaysSinceLastMatch', 'LoserDaysSinceLastMatch'),
    ('forma_last10', 'WinnerFormaLast10', 'LoserFormaLast10'),
    ('career_winrate_before', 'WinnerCareerWinRateBefore', 'LoserCareerWinRateBefore'),
    ('h2h_win_rate_before', 'WinnerH2HWinRateBefore', 'LoserH2HWinRateBefore'),
    ('debut_razina', 'WinnerDebutRazina', 'LoserDebutRazina'),
    ('debut_broj_liga', 'WinnerDebutBrojLiga', 'LoserDebutBrojLiga'),
    ('debut_broj_igraca', 'WinnerDebutBrojIgraca', 'LoserDebutBrojIgraca'),
    ('debut_relative_razina', 'WinnerDebutRelativeRazina', 'LoserDebutRelativeRazina'),
    ('season_match_count_before', 'WinnerSeasonMatchCountBefore', 'LoserSeasonMatchCountBefore'),
    ('home_share_before', 'WinnerHomeShareBefore', 'LoserHomeShareBefore'),
    ('age', 'WinnerAge', 'LoserAge'),
    ('stb_win_rate_before', 'WinnerSTBWinRateBefore', 'LoserSTBWinRateBefore'),
    ('stb_count_before', 'WinnerSTBCountBefore', 'LoserSTBCountBefore'),
    ('decider_win_rate_before', 'WinnerDeciderWinRateBefore', 'LoserDeciderWinRateBefore'),
    ('decider_count_before', 'WinnerDeciderCountBefore', 'LoserDeciderCountBefore'),
]

SYMMETRIC_COLS = [
    ('h2h_total_before', 'H2HTotalBefore'),
    ('liga_koeficjent', 'LigaKoeficjent'),
    ('liga_tip', 'LigaTip'),
    ('broj_liga_u_sezoni', 'BrojLigaUSezoni'),
    ('broj_igraca_ukupno', 'BrojIgracaUkupno'),
    ('broj_igraca_uligi', 'BrojIgracaULigi'),
    ('relative_razina_by_liga_count', 'RelativeRazina_ByLigaCount'),
    ('relative_razina_by_player_count', 'RelativeRazina_ByPlayerCount'),
    ('relative_razina_pct', 'RelativeRazinaPct'),
]

# every ASYMMETRIC_PAIRS / SYMMETRIC_COLS name must have an audit + tag entry
missing_tags = [n for n, *_ in ASYMMETRIC_PAIRS if n not in FEATURE_TAGS] + \
               [n for n, _ in SYMMETRIC_COLS if n not in FEATURE_TAGS]
assert not missing_tags, f"Untagged features: {missing_tags}"
print("OK -- every asymmetric/symmetric feature has a leakage-audit + tag entry.")


In [ ]:
META_COLS = ['SusretId', 'Datum', 'SezonaId', 'LigaId']

def build_symmetric_frame(df, asym_pairs, sym_cols, meta_cols):
    frames = []
    for is_winner_perspective in (True, False):
        part = pd.DataFrame(index=df.index)
        for m in meta_cols:
            part[m] = df[m]
        part['PlayerId'] = df['WinnerId'] if is_winner_perspective else df['LoserId']
        part['OpponentId'] = df['LoserId'] if is_winner_perspective else df['WinnerId']
        part['Label'] = 1 if is_winner_perspective else 0
        part['Perspective'] = 'winner_pov' if is_winner_perspective else 'loser_pov'

        for name, wcol, lcol in asym_pairs:
            player_col = df[wcol] if is_winner_perspective else df[lcol]
            opp_col = df[lcol] if is_winner_perspective else df[wcol]
            part[f'{name}_diff'] = player_col - opp_col
            part[f'player_{name}'] = player_col       # audit/debug only, not a model input
            part[f'opponent_{name}'] = opp_col        # audit/debug only, not a model input

        for name, col in sym_cols:
            part[name] = df[col]

        frames.append(part)

    return pd.concat(frames, ignore_index=True)

sym_frame = build_symmetric_frame(df, ASYMMETRIC_PAIRS, SYMMETRIC_COLS, META_COLS)
print(sym_frame.shape, '(expected:', 2 * len(df), 'rows)')
sym_frame.head()


In [ ]:
print(f"Total rows: {len(sym_frame):,} (2 x {len(df):,} matches)")
print(sym_frame['Label'].value_counts(normalize=True))
assert len(sym_frame) == 2 * len(df)
assert (sym_frame['Label'].value_counts(normalize=True) == 0.5).all(), "Label is not exactly 50/50!"
print("OK -- exactly 50/50 class balance by construction.")


## Step 10 — Time-based train/test split marker (Part 2.4)

Sort MATCHES (not the duplicated rows) by `Datum ASC, SusretId ASC`. First 80% -> train, last 20% -> test. Both perspective-rows of the same match are assigned to the SAME split -- otherwise the same match's information would leak across the train/test boundary. **Never a random split** -- this is a time series (Elo, H2H, forma are all derived from match order; a random split would let future-derived signal leak into training).

In [ ]:
match_order = df[['SusretId', 'Datum']].sort_values(['Datum', 'SusretId']).reset_index(drop=True)
n_matches = len(match_order)
split_idx = int(n_matches * 0.8)
split_date = match_order.iloc[split_idx]['Datum']
train_susret_ids = set(match_order.iloc[:split_idx]['SusretId'])

sym_frame['Split'] = np.where(sym_frame['SusretId'].isin(train_susret_ids), 'train', 'test')

print(f"Total matches: {n_matches:,}  ->  train: {split_idx:,}  test: {n_matches - split_idx:,}")
print(f"Split date (first test-set match date): {split_date.date()}")
print()
print(sym_frame['Split'].value_counts())
print(sym_frame.groupby('Split')['Datum'].agg(['min', 'max']))


## Step 11 — Model-input column sets for Experiment A / B (Part 2.1/2.2)

`MODEL_INPUT_COLS` = all `*_diff` columns + all symmetric columns -- this is the intended P3
feature matrix. `AUDIT_ONLY_COLS` (`player_<x>`/`opponent_<x>`) are retained in the saved frame for
transparency/debugging but are excluded from `EXP_A_COLS`/`EXP_B_COLS`.


In [ ]:
COLUMN_TAGS = {}
for name, _, _ in ASYMMETRIC_PAIRS:
    tag = FEATURE_TAGS[name]
    COLUMN_TAGS[f'{name}_diff'] = tag
for name, _ in SYMMETRIC_COLS:
    COLUMN_TAGS[name] = FEATURE_TAGS[name]

MODEL_INPUT_COLS = [f'{name}_diff' for name, _, _ in ASYMMETRIC_PAIRS] + [name for name, _ in SYMMETRIC_COLS]
AUDIT_ONLY_COLS = [f'player_{name}' for name, _, _ in ASYMMETRIC_PAIRS] + [f'opponent_{name}' for name, _, _ in ASYMMETRIC_PAIRS]

elo_features = [c for c in MODEL_INPUT_COLS if COLUMN_TAGS[c] == 'ELO']
elo_sub_features = [c for c in MODEL_INPUT_COLS if COLUMN_TAGS[c] == 'ELO-substitute']
neutral_features = [c for c in MODEL_INPUT_COLS if COLUMN_TAGS[c] == 'neutral']

print(f"ELO ({len(elo_features)}): {elo_features}")
print(f"ELO-substitute ({len(elo_sub_features)}): {elo_sub_features}")
print(f"neutral ({len(neutral_features)}): {neutral_features}")

EXP_A_COLS = [c for c in MODEL_INPUT_COLS if COLUMN_TAGS[c] != 'ELO']  # ELO-free
EXP_B_COLS = MODEL_INPUT_COLS                                          # everything

print(f"\nExperiment A (ELO-free) feature count: {len(EXP_A_COLS)}")
print(f"Experiment B (ELO-included) feature count: {len(EXP_B_COLS)}")

# Re-verify: no ELO-substitute feature's SOURCE column touches an Elo column.
for name, wcol, lcol in ASYMMETRIC_PAIRS:
    if FEATURE_TAGS[name] == 'ELO-substitute':
        assert 'Elo' not in wcol and 'Elo' not in lcol, f"{name} ({wcol}/{lcol}) touches an Elo column!"
for name, col in SYMMETRIC_COLS:
    if FEATURE_TAGS[name] == 'ELO-substitute':
        assert 'Elo' not in col, f"{name} ({col}) touches an Elo column!"
print("\nRe-verified: no ELO-substitute feature's source column reads EloBefore/EloAfter.")


## Step 12 — Save engineered frame + tagging dict

In [ ]:
sym_frame.to_parquet(DATA_DIR / 'features.parquet', index=False)

tagging_payload = {
    'feature_tags_base': FEATURE_TAGS,
    'column_tags': COLUMN_TAGS,
    'model_input_cols': MODEL_INPUT_COLS,
    'audit_only_cols': AUDIT_ONLY_COLS,
    'experiment_a_cols': EXP_A_COLS,
    'experiment_b_cols': EXP_B_COLS,
    'split_date': str(split_date.date()),
    'n_matches_train': int(split_idx),
    'n_matches_test': int(n_matches - split_idx),
    'shrinkage': {'h2h_prior': H2H_PRIOR, 'clutch_prior': CLUTCH_PRIOR},
    'godiste_bounds': {'lower': GODISTE_LOWER, 'upper': GODISTE_UPPER},
}
with open(DATA_DIR / 'feature_tags.json', 'w', encoding='utf-8') as f:
    json.dump(tagging_payload, f, indent=2, ensure_ascii=False)

print('Saved:')
print(' -', DATA_DIR / 'features.parquet', f'({sym_frame.shape[0]:,} rows x {sym_frame.shape[1]} cols)')
print(' -', DATA_DIR / 'feature_tags.json')


## Step 13 — Final verification

In [ ]:
print('Final sym_frame shape:', sym_frame.shape)
print()
print('Label balance:')
print(sym_frame['Label'].value_counts(normalize=True))
print()
print('Split sizes:')
print(sym_frame['Split'].value_counts())
print()
print('NaN share per model-input column (sorted descending):')
print(sym_frame[MODEL_INPUT_COLS].isna().mean().sort_values(ascending=False))


## Summary

- **Notebook-derived features added:** `season_match_count_before`, `home_share_before` (+diff),
  `age` (+diff), `stb_win_rate_before`/`stb_count_before` (+diffs), `decider_win_rate_before`/
  `decider_count_before` (+diffs), `relative_razina_by_liga_count`, `relative_razina_by_player_count`,
  `relative_razina_pct`, `debut_relative_razina` (+diff), plus the shrunk `h2h_win_rate_before`
  (+diff) and `elo_confidence` (+diff) meta-features from the feature-tagging step above.
- **Leakage audit:** 28 base features audited, 0 read the outcome of the match being predicted,
  every feature has an explicit point-in-time rule or a documented roster/signup exemption.
- **Tagging:** feature tags saved to `feature_tags.json`, sliceable into `experiment_a_cols`
  (ELO-free) vs `experiment_b_cols` (all) for the modeling notebooks.
- **Duplicate-and-flip:** `2 x n_matches` rows, exactly 50/50 `Label` balance by construction.
- **Time-based split:** first 80% of matches (by `Datum`, `SusretId` tiebreak) -> train, last 20%
  -> test; both perspective-rows of a match always share the same split.
- **Output:** `../data/features.parquet` (gitignored -- contains real `IgracId`s) and
  `../data/feature_tags.json` (no PII -- just feature-name -> tag metadata).